<a href="https://colab.research.google.com/github/Reinhart-py/Universal-Archive-Extractor-drive/blob/main/Universal_Archive_Extractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# @title 📦 Universal Archive Extractor (Zip, Rar, 7z, Tar, etc.)
import os
import sys
from IPython.display import clear_output

# 1. Install required packages automatically
print("⚙️ Installing necessary dependencies (this takes a few seconds)...")
os.system("apt-get update > /dev/null 2>&1")
os.system("apt-get install -y unrar p7zip-full p7zip-rar > /dev/null 2>&1")
os.system("pip install rarfile py7zr tqdm > /dev/null 2>&1")
clear_output()

import zipfile
import tarfile
import rarfile
import py7zr
from tqdm.notebook import tqdm

# @markdown ### 📂 Input your file or folder path here:
INPUT_PATH = "/content/drive/MyDrive/Backup /Tools creaked/50+ tools/Windows Tools.rar" # @param {type:"string"}

def extract_archive(file_path):
    filename = os.path.basename(file_path)
    name, ext = os.path.splitext(filename)
    ext = ext.lower()

    # Create an output folder matching the archive's name (in the same directory)
    target_dir = os.path.join(os.path.dirname(file_path), name)
    os.makedirs(target_dir, exist_ok=True)

    try:
        # ZIP FILES
        if ext == '.zip':
            with zipfile.ZipFile(file_path, 'r') as zf:
                members = zf.infolist()
                total_size = sum(m.file_size for m in members)
                with tqdm(total=total_size, desc=f"Extracting {filename}", unit='B', unit_scale=True, unit_divisor=1024, leave=False) as pbar:
                    for member in members:
                        zf.extract(member, target_dir)
                        pbar.update(member.file_size)

        # TAR / GZ / BZ2 FILES
        elif ext in ['.tar', '.gz', '.bz2', '.tgz', '.xz']:
            with tarfile.open(file_path, 'r') as tf:
                members = tf.getmembers()
                total_size = sum(m.size for m in members)
                with tqdm(total=total_size, desc=f"Extracting {filename}", unit='B', unit_scale=True, unit_divisor=1024, leave=False) as pbar:
                    for member in members:
                        tf.extract(member, target_dir)
                        pbar.update(member.size)

        # RAR FILES
        elif ext == '.rar':
            with rarfile.RarFile(file_path, 'r') as rf:
                members = rf.infolist()
                total_size = sum(m.file_size for m in members)
                with tqdm(total=total_size, desc=f"Extracting {filename}", unit='B', unit_scale=True, unit_divisor=1024, leave=False) as pbar:
                    for member in members:
                        rf.extract(member, target_dir)
                        pbar.update(member.file_size)

        # 7Z FILES
        elif ext == '.7z':
            with py7zr.SevenZipFile(file_path, 'r') as szf:
                # 7z uses "solid" compression, so we extract all at once instead of file-by-file
                print(f"⏳ Extracting {filename} (7z solid archive format - progress bar updates at completion)...")
                szf.extractall(path=target_dir)

        else:
            print(f"⚠️ Unsupported format for {filename}: {ext}")
            # Clean up the empty directory created for it
            if not os.listdir(target_dir):
                os.rmdir(target_dir)
            return False

        print(f"✅ Successfully extracted: {filename} ➔ {target_dir}")
        return True

    except Exception as e:
        # Error handling: Catches corruption/errors but allows the script to continue to the next file
        print(f"❌ Error extracting {filename}: {str(e)}")
        return False

def main():
    input_path = INPUT_PATH.strip()

    if not os.path.exists(input_path):
        print(f"❌ Error: Path does not exist -> {input_path}")
        return

    supported_exts = ['.zip', '.rar', '.7z', '.tar', '.gz', '.bz2', '.tgz', '.xz']

    # SCENARIO 1: It is a single file
    if os.path.isfile(input_path):
        ext = os.path.splitext(input_path)[1].lower()
        if ext in supported_exts:
            print(f"🎯 Target is a single file. Starting extraction...\n")
            extract_archive(input_path)
        else:
            print(f"❌ Unsupported file type: {input_path}")

    # SCENARIO 2: It is a directory
    elif os.path.isdir(input_path):
        print(f"📁 Target is a directory. Scanning for archives in: {input_path}")
        files_to_extract = []

        # Scan directory for all supported archives
        for root, dirs, files in os.walk(input_path):
            for f in files:
                if os.path.splitext(f)[1].lower() in supported_exts:
                    files_to_extract.append(os.path.join(root, f))

        if not files_to_extract:
            print(f"⚠️ No supported archives found in {input_path}")
            return

        print(f"🔍 Found {len(files_to_extract)} archives. Starting continuous extraction...\n")

        success_count = 0
        error_count = 0

        # Loop through each file one after another
        for idx, file_path in enumerate(files_to_extract, 1):
            print(f"\n[{idx}/{len(files_to_extract)}] Processing: {os.path.basename(file_path)}")
            if extract_archive(file_path):
                success_count += 1
            else:
                error_count += 1

        print("\n" + "="*45)
        print(f"🎉 Extraction Process Finished!")
        print(f"✅ Successfully extracted : {success_count}")
        if error_count > 0:
            print(f"❌ Failed / Corrupted   : {error_count}")
        print("="*45)

if __name__ == "__main__":
    main()

🎯 Target is a single file. Starting extraction...



Extracting Windows Tools.rar:   0%|          | 0.00/8.26G [00:00<?, ?B/s]